# Laden von OSM mittels osmium
- https://duckdb.org/community_extensions/extensions/osmium

In [ ]:
import duckdb
import os
import time
import requests

## Abfrage der aktuellen OSM-Daten (Standard nicht älter als 4 Tage)

In [ ]:
url = 'https://download.geofabrik.de/europe/germany/niedersachsen-latest.osm.pbf'
output_path = "downloads/niedersachsen-latest.osm.pbf"

In [ ]:
if os.path.exists(output_path):
    mtime = os.path.getmtime(output_path)
    age_hours = (time.time() - mtime) / 3600
    print(f"File '{output_path}' was last modified {age_hours:.1f} hours ago.")
else:
    print(f"File '{output_path}' does not exist.")

In [ ]:
# Download a pbf file to the downloads folder
# If the file is older than 96 hours, re-download it
if age_hours > 96:
    response = requests.get(url)
    with open(output_path, "wb") as f:
        f.write(response.content)
    print(f"Downloaded file to {output_path}")

## Start der Datenbank und Beispielabfragen

In [ ]:
duck = duckdb.connect()

In [ ]:
duck.sql("""
SELECT extension_name, installed, description
FROM duckdb_extensions();
""").df()

In [ ]:
duck.sql(""" install osmium from community;
load osmium;
install spatial;
load spatial;
SET geometry_always_xy = true;
-- INSTALL openfgdb FROM community;
-- LOAD openfgdb;

""")

In [ ]:
duck.sql("from st_drivers()").df()

In [ ]:
duck.sql(f"""

LOAD osmium;
SELECT id, tags['name'] AS name, geometry
FROM '{output_path}'
WHERE kind = 'node' AND tags['place'] = 'city';

""")

## Import GTFS

In [ ]:
gtfs_stops = '../gtfs/gtfs_dhid/stops.txt'
gtfs_stop_times = '../gtfs/gtfs_dhid/stop_times.txt'
gtfs_trips = '../gtfs/gtfs_dhid/trips.txt'
gtfs_routes = '../gtfs/gtfs_dhid/routes.txt'
gtfs_shapes = '../gtfs/gtfs_dhid/shapes.txt'
gtfs_agency = '../gtfs/gtfs_dhid/agency.txt'

In [ ]:
duck.sql(f"""
CREATE or replace TABLE stops AS select *, st_makepoint(stop_lon, stop_lat) as geom from read_csv('{gtfs_stops}');
CREATE or replace TABLE stop_times AS select * from read_csv('{gtfs_stop_times}',delim=',', ignore_errors=true,
columns = {{
        'trip_id' : 'VARCHAR',
        'stop_id' : 'VARCHAR',
        'stop_sequence' : 'INTEGER',
        'pickup_type' : 'VARCHAR',
        'drop_off_type' : 'VARCHAR',
        'stop_headsign' : 'VARCHAR',

        'arrival_time' : 'VARCHAR',
        'departure_time' : 'VARCHAR',
        
}});
CREATE or replace TABLE trips AS select * from read_csv('{gtfs_trips}', delim=',', store_rejects = true,
columns = {{ 
        'route_id' : 'VARCHAR',       
        'service_id' : 'VARCHAR',       
            
        'trip_id' : 'VARCHAR',       
        'trip_headsign' : 'VARCHAR',       
        'trip_short_name': 'VARCHAR',        
        'direction_id': 'VARCHAR',        
        'block_id': 'VARCHAR',        
        'shape_id': 'VARCHAR'  ,      
        'wheelchair_accessible': 'VARCHAR' ,       
        'bikes_allowed': 'VARCHAR'        
    }});
CREATE or replace TABLE routes AS select * from read_csv('{gtfs_routes}');
CREATE or replace TABLE shapes AS select * from read_csv('{gtfs_shapes}');
CREATE or replace TABLE agency AS select * from read_csv('{gtfs_agency}');
         """)

In [ ]:
duck.sql("""
create or replace table verlauf as

select * exclude(st.pickup_type, st.drop_off_type, s.stop_id, st.stop_headsign, t.trip_headsign,s.stop_lat, 
s.stop_lon ,t.trip_id, arrival_time, t.block_id, platform_code, t.wheelchair_accessible, t.service_id, t.bikes_allowed, 
departure_time, stop_desc, zone_id, wheelchair_boarding, location_type, parent_station,
r.route_long_name, r.route_type, r.route_text_color, r.route_desc, r.route_color, 
a.agency_url, a.agency_timezone, a.agency_lang, a.agency_phone, s.geom)
, st_transform(geom, 'EPSG:4326', 'EPSG:25832') AS geometry
from stop_times st
join stops s on st.stop_id = s.stop_id
join trips t on st.trip_id = t.trip_id
join routes r on t.route_id = r.route_id
join agency a on r.agency_id = a.agency_id

where a.agency_id not in (66,81,106,134,135,136,138,140,143,146,149,150,152,176,
216,218,221,226,231,253,257,259,266,271,286,
326,396,
401,402,403,
546,547,548,549,550,552,553,554,
605,606,607,608,615,620,685,690,
720,760,780,795,
810,815,
969,
1130,1322,1935,1950,19902160,2205,2240,2400,2506,2550,2555);



""")

In [ ]:
duck.sql("alter table verlauf alter geometry type GEOMETRY;")

### Export der Wege nach Parquet/FGB

In [ ]:
duck.sql(f"""

LOAD osmium;

copy (

SELECT id, tags['name'] AS name, tags['highway'] AS highway, tags['foot'] AS foot, tags['level'] AS level, tags['layer'] AS layer,
tags['access'] AS access, tags['bus'] AS bus, tags['bus_allowed'] AS bus_allowed, tags['foot_allowed'] AS foot_allowed,
 geometry
FROM '{output_path}'
WHERE kind = 'line'
  AND tags['highway'] IS NOT NULL)

  to 'out/ways.geoparquet' (FORMAT 'parquet', COMPRESSION 'snappy')

""")

In [ ]:
duck.sql(f"""

LOAD osmium;

copy 
    (
    SELECT id, tags['name'] AS name, tags['highway'] AS highway, tags['foot'] AS foot, tags['level'] AS level, tags['layer'] AS layer,
    tags['access'] AS access, tags['bus'] AS bus, tags['bus_allowed'] AS bus_allowed, tags['foot_allowed'] AS foot_allowed,
    tags['oneway'] AS oneway, tags['maxspeed'] AS maxspeed, tags['lanes'] AS lanes, tags['surface'] AS surface,
    tags['bicycle'] AS bicycle, 
    st_transform(geometry, 'EPSG:4326', 'EPSG:25832') AS geometry
    FROM '{output_path}'
    WHERE kind = 'line'
        AND tags['highway'] IS NOT NULL
    )

 TO 'out/ways_na4.gdb' (FORMAT gdal, 
                        DRIVER 'OpenFileGDB', 
                        SRS 'EPSG:25832', 
                        LAYER_NAME 'ways_na', 
                        GEOMETRY_TYPE 'LINESTRING',
                                        LAYER_CREATION_OPTIONS (
                                                'FEATURE_DATASET=osm', 
                                                'LAYER_NAME=highways_nds_sel', 
                                                'LAYER_ALIAS=auswahl_osm'
                                                ));
""")

In [ ]:
duck.sql("from verlauf limit 5")

In [ ]:
duck.sql(f"""

LOAD osmium;

copy 
    (
    from verlauf
    )

 TO 'out/verlauf_gtfs.gdb' (FORMAT gdal, 
                        DRIVER 'OpenFileGDB', 
                        SRS 'EPSG:25832', 
                        LAYER_NAME 'verlauf_gtfs', 
                        GEOMETRY_TYPE 'POINT',
                                        LAYER_CREATION_OPTIONS (
                                                'FEATURE_DATASET=gtfs', 
                                                'LAYER_NAME=verlauf', 
                                                'LAYER_ALIAS=verlauf'
                                                ));
""")